In [ ]:
import asyncio
import datetime
import json
import requests
import os
from dotenv import load_dotenv
from typing import List, Sequence
from rich.console import Console
from rich.text import Text
from rich.markdown import Markdown
from autogen_agentchat.agents import AssistantAgent, BaseChatAgent, UserProxyAgent
from autogen_agentchat.base import Response, TaskResult
from autogen_agentchat.messages import ChatMessage, StopMessage, TextMessage
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient


# Tool to search the web using Bing
async def get_bing_snippet(query: str) -> str:
    #Perform a web search using the Bing Web Search API.
    # Set the parameters for the API request.
    count = 3       # Number of search results to return
    params = {
        'q': query,
        'count': count,
    }

    # Set the headers for the API request, including the subscription key.
    headers = {
        'Ocp-Apim-Subscription-Key': bing_api_key,
    }

    # Make the API request.
    response = requests.get(bing_endpoint, params=params, headers=headers)
    
    # Check if the request was successful (HTTP status code 200).
    if response.status_code == 200:
        search_results = response.json()
        # Extract and structure the search results.
        results_list = []
        for result in search_results['webPages']['value']:
            result_tuple = (result['name'], result['snippet'], result['url'])
            results_list.append(result_tuple)
        return json.dumps(results_list)
    else:
        error = f"Error: {response.status_code} - {response.text}"
        print(error)
        return error


async def main() -> None:
    # Define agents
    user_proxy = UserProxyAgent("User")

    web_search_agent = AssistantAgent(
        name="web_search_agent",
        description="An agent who can search the web to conduct research and answer open questions",
        model_client=AzureOpenAIChatCompletionClient(
            model=azure_model_deployment,
            api_version=azure_api_version,
            azure_endpoint=azure_oai_endpoint,
            api_key=azure_oai_key,
            model_capabilities={
                "vision": True,
                "function_calling": True,
                "json_output": True,
            },
        ),
        tools=[get_bing_snippet],
    )
    
    editor_agent = AssistantAgent(
        name = "editor", 
        description="An expert editor of written articles who can read an article and make suggestions for improvements and additional topics that should be researched",
        model_client=AzureOpenAIChatCompletionClient(
            model=azure_model_deployment,
            api_version=azure_api_version,
            azure_endpoint=azure_oai_endpoint,
            api_key=azure_oai_key,
            model_capabilities={
                "vision": True,
                "function_calling": True,
                "json_output": True,
            },
        ), 
        system_message="You are an expert editor.  You carefully read an article and make suggestions for improvements and suggest additional topics that should be researched to improve the article quality."
    )

    verifier_agent = AssistantAgent(
        name = "verifier_agent", 
        description="A responsible agent who will verify the facts and ensure that the article is accurate and well-written",
        model_client=AzureOpenAIChatCompletionClient(
            model=azure_model_deployment,
            api_version=azure_api_version,
            azure_endpoint=azure_oai_endpoint,
            api_key=azure_oai_key,
            model_capabilities={
                "vision": True,
                "function_calling": True,
                "json_output": True,
            },
        ), 
        tools=[get_bing_snippet],
        system_message="You are responsible for ensuring the article's accuracy.  You should use the Bing tool to search the internet to verify any relevant facts, and explicitly approve or reject the article based on accuracy, giving your reasoning. You can ask for rewrites if you find inaccuracies."
    )

    writer_assistant = AssistantAgent(
        name = "writer_assistant", 
        description="A high-quality journalist agent who excels at writing a first draft of an article as well as revising the article based on feedback from the other agents",
        model_client=AzureOpenAIChatCompletionClient(
            model=azure_model_deployment,
            api_version=azure_api_version,
            azure_endpoint=azure_oai_endpoint,
            api_key=azure_oai_key,
            model_capabilities={
                "vision": True,
                "function_calling": True,
                "json_output": True,
            },
        ), 
        system_message="You are a high-quality journalist agent who excels at writing a first draft of an article as well as revising the article based on feedback from the other agents.  Do not just write bullet points on how you would write the article, but actually write it.  You can also ask for research to be conducted on certain topics. "
    )

    orchestrator_agent = AssistantAgent(
        name = "orchestrator_agent", 
        description="Team leader who verifies when the article is complete and meets all requirements",
        model_client=AzureOpenAIChatCompletionClient(
            model=azure_model_deployment,
            api_version=azure_api_version,
            azure_endpoint=azure_oai_endpoint,
            api_key=azure_oai_key,
            model_capabilities={
                "vision": True,
                "function_calling": True,
                "json_output": True,
            },
        ), 
        system_message="You are a leading a journalism team that conducts research to craft high-quality articles.  You ensure that the output contains an actual well-written article, not just bullet points on what or how to write the article.  If the article isn't to that level yet, ask the writer for a rewrite.  If the team has written a strong article with a clear point that meets the requirements, and has been reviewed by the editor, and has been fact-checked and approved by the verifier agent, and approved by the user, then reply 'TERMINATE'.  Otherwise state what condition has not yet been met."
    )


    # Define termination condition
    termination = TextMentionTermination("TERMINATE", ["orchestrator_agent"])

    # Define a team
    agent_team = SelectorGroupChat(
        [writer_assistant, web_search_agent, editor_agent, verifier_agent, user_proxy, orchestrator_agent,], 
        model_client=AzureOpenAIChatCompletionClient(
            model=azure_model_deployment,
            api_version=azure_api_version,
            azure_endpoint=azure_oai_endpoint,
            api_key=azure_oai_key,
            model_capabilities={
                "vision": True,
                "function_calling": True,
                "json_output": True,
            },
        ),
        termination_condition=termination
    )

    # Define the task prompt
    task_prompt = "Ask the user to describe the article they want to write. They can include some starting bullet points if they want. Today's date is " + str(datetime.date.today())

    # Run the team and stream messages
    stream = agent_team.run_stream(task=task_prompt)
    console = Console()
    async for response in stream:
        #print(response)
        text = Text()
        if not isinstance(response, TaskResult):
            # Print the agent name in color
            text.append(response.source, style="bold magenta")
            text.append(": ")
            console.print(text)
            if isinstance(response, str):
                md = Markdown(response.content)
                console.print(md)
            else:
                console.print(response.content)
        else:
            console.print(response.stop_reason)



# Load env variables
load_dotenv()
azure_oai_endpoint = os.getenv("AZURE_OPENAI_API_ENDPOINT")
azure_oai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_model_deployment = os.getenv("AZURE_MODEL_DEPLOYMENT")
azure_api_version = os.getenv("AZURE_OPENAI_API_VERSION")
bing_endpoint = os.getenv("BING_ENDPOINT")
bing_api_key = os.getenv("BING_API_KEY") 

# Run
asyncio.run(main())

In [ ]:
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.base import TaskResult
from autogen_agentchat.conditions import ExternalTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Create an OpenAI model client.
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-2024-08-06",
    # api_key="sk-...", # Optional if you have an OPENAI_API_KEY env variable set.
)

# Create the primary agent. That is in charge of creating supporting materials of a MSFT course based on the Trainers preferences. For each module, it will call on it's agents and final role will be to compile all resources into a word document 
orchestrator_agent = AssistantAgent(
    "primary",
    model_client=model_client,
    system_message="You are a helpful AI assistant.",
)

#agent that is in charge of retrieving MS learn modules/ content
mslearn_gather_agent = AssistantAgent()

#agent in charge of retrieving PowerPOint based on Existing content
mct_presentation_gather_agent = AssistantAgent()

#agent in charge of learning about trainers prior experiences based on one-note/ certification/ context memory from a .txt file. They will approach the task from the perspective of a microsoft certified trainer and check if the module includes appropriate content. Also Suggests interactive elements (polls, breakout prompts, etc.) to boost learner participation.
#presenter_profiler_agent = AssistantAgent()

#localization_agent – Adapts content for different regions/languages (especially useful in EMEA).
localisation_agent = AssistantAgent()

#module_agent - in charge of creating a summary and training points based on all the materials gathered 
module_agent = AssistantAgent()

#agent in charge of creating a boiler plate code that is in line with the module_agents summary + latest learn documentatIon
demo_agent = AssistantAgent()

# learner agent that takes in the role of an learner from an engagement perspective. E.g. They assume the target audience of the role from a beginners/junior role standpoint
jnr_learner_agent = AssistantAgent()

# seniour learner agent that takes in the role of an learner from an engagement perspective. E.g. They assume the target audience of the role from an experienced/senior role standpoint

snr_learner_agent = AssistantAgent()

#classroom agent <- checks the current teaching recommendations for the course and designs a schedule based on the day and module demo
classroom_agent()

#agent that generates quizzes/knowledge checks
assessment_agent = AssistantAgent()

# Create the critic agent. Evaluate whether the current output per module covers all the basics placing weighting on the powerpoint 
critic_agent = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message="Provide constructive feedback. Respond with 'APPROVE' to when your feedbacks are addressed.",
)

# Define a termination condition that stops the task if the critic approves.
text_termination = TextMentionTermination("APPROVE")

# Create a team with the primary and critic agents.
team = RoundRobinGroupChat([module_agent, mslearn_gather_agent, mct_presentation_gather_agent, presenter_profiler_agent], termination_condition=text_termination)


In [ ]:
result = await team.run(task="Write a short poem about the fall season.")
